파일을 sc.textFile()로 읽어서 다음을 계산하세요.
• 첫 번째 정수 쌍의 첫 번째 원소
• 모든 정수 쌍의 첫 번째 원소 중 최댓값
• 모든 정수 쌍의 첫 번째 원소 중 최솟값
• 첫 번째 원소가 1인 정수 쌍의 개수
• 첫 번째 원소가 2인 정수 쌍의 개수
• 이 다섯 개의 숫자를 모두 더한 뒤 출력하세요.

In [2]:
import os
import sys
import urllib.request
from pyspark.sql import SparkSession

# Windows VS Code에서 발생했던 Python Worker 연결 시간 초과 오류에 대응
# Python 실행 경로를 지정하고 Spark Driver의 로컬 통신 주소를 설정
if os.name == "nt":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark_builder = SparkSession.builder.appName("Q2_IntPair").master("local[*]")
if os.name == "nt":
    spark_builder = (spark_builder
                     .config("spark.driver.host", "127.0.0.1")
                     .config("spark.driver.bindAddress", "127.0.0.1"))

spark = spark_builder.getOrCreate()
sc = spark.sparkContext

# 과제에서 지정한 파일을 다운로드하고 sc.textFile()로 RDD 생성
url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt"
urllib.request.urlretrieve(url, "int-pair-input.txt")
lines = sc.textFile("int-pair-input.txt")

# 빈 줄을 제외하고, 문자열 '(x, y)'를 정수 쌍 [x, y]로 변환 (Transformation)
pairs_rdd = (lines
    .filter(lambda line: line.strip() != "")
    .map(lambda line: [int(v) for v in line.replace("(", "")
                                               .replace(")", "")
                                               .replace(",", " ").split()]))

# 모든 정수 쌍에서 첫 번째 원소만 추출 (Transformation)
first_elems_rdd = pairs_rdd.map(lambda pair: pair[0])

# 수업에서 배운 Action을 이용해 필요한 숫자만 Driver로 반환
# RDD 전체를 collect()하여 Python for문으로 처리하지 않음
v1 = pairs_rdd.first()[0]                             # 첫 번째 정수 쌍의 첫 원소
v2 = first_elems_rdd.reduce(lambda a, b: max(a, b))   # 최댓값
v3 = first_elems_rdd.reduce(lambda a, b: min(a, b))   # 최솟값
v4 = first_elems_rdd.filter(lambda x: x == 1).count() # 첫 원소가 1인 쌍의 개수
v5 = first_elems_rdd.filter(lambda x: x == 2).count() # 첫 원소가 2인 쌍의 개수

# 다섯 개의 숫자 결과만 합산 (RDD 데이터를 Driver로 가져와 반복 처리하지 않음)
final_answer = v1 + v2 + v3 + v4 + v5

print(f"1. 첫 번째 정수 쌍의 첫 원소: {v1}")
print(f"2. 첫 번째 원소 최댓값: {v2}")
print(f"3. 첫 번째 원소 최솟값: {v3}")
print(f"4. 첫 번째 원소가 1인 개수: {v4}")
print(f"5. 첫 번째 원소가 2인 개수: {v5}")
print(f"\n최종 합계: {final_answer}")


1. 첫 번째 정수 쌍의 첫 원소: 2
2. 첫 번째 원소 최댓값: 32
3. 첫 번째 원소 최솟값: 0
4. 첫 번째 원소가 1인 개수: 31
5. 첫 번째 원소가 2인 개수: 28

최종 합계: 93
